# BirdCLEF+ 2026 — Submission Notebook

Self-contained inference notebook. No repo imports — all model code inlined.

**Model options** (set `MODEL_NAME` env var or edit Cell 2):
- `pretrained_transformer` — ViT-Small fine-tuned, Kaggle track (sc_auc checkpoint, K=234) ✓ default
- `cnn_transformer` — ResNet18 + Transformer, Kaggle track (sc_auc checkpoint, K=234)

Kaggle track checkpoints (`best_*_kaggle.pt`) are selected on soundscape val AUC. Report track
checkpoints (`best_*_report.pt`) use clean-audio val AUC and are for RESULTS.md / ablations only.

---

**Setup checklist — do this before running:**

**1. Attach competition data**
- Notebook sidebar → Add Data → Competition Data → search `birdclef-2026` → Add
- This mounts at `/kaggle/input/birdclef-2026/` (required — Cell 2 auto-detects this path)

**2. Attach model weights**
- Model weights are at [kaggle.com/datasets/flixwg/aml2026-model-weights](https://www.kaggle.com/datasets/flixwg/aml2026-model-weights)
- Notebook sidebar → Add Data → Your Datasets → `aml2026-model-weights` → Add
- Weights mount at `/kaggle/input/datasets/flixwg/aml2026-model-weights/`

**3. Session settings**
- Accelerator: **None (CPU)**
- Internet: **Off**
- Do NOT set `DATA_ROOT` environment variable — Cell 2 auto-detects the correct path

---

**Save Version vs Submit to Competition**

BirdCLEF 2026 is a sealed code competition. During **Save Version**, `test_soundscapes/` is empty — Cell 7 falls back to ~20 train files so the notebook still produces a valid `submission.csv` (format check only). The log will show train `row_id`s like `BC2026_Train_*` and shape `(240, 235)`.

When you click **Submit to Competition**, Kaggle re-runs this notebook with the hidden test set mounted. That re-run's `submission.csv` is what gets scored on the leaderboard — not the save-run output.

In [1]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import glob
import os

import librosa
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        desc = kwargs.get('desc', '')
        items = list(iterable)
        for i, item in enumerate(items):
            print(f"{desc} {i+1}/{len(items)}", end='\r')
            yield item
        print()

print(f"torch={torch.__version__}  timm={timm.__version__}  librosa={librosa.__version__}")

torch=2.10.0+cpu  timm=1.0.25  librosa=0.11.0


In [2]:
# ── Cell 2: Config — edit here or set env vars ────────────────────────────────
#
# MODEL_NAME:  which checkpoint to load
#   "pretrained_transformer"  →  ViT-Small Kaggle track (sc_auc, K=234)  [default]
#   "cnn_transformer"         →  ResNet18+Transformer Kaggle track (sc_auc, K=234)
#
# Checkpoint filenames use the _kaggle suffix produced by:
#   python train.py --model <name> --checkpoint_metric sc_auc --taxonomy_csv taxonomy.csv --tag kaggle
#
MODEL_NAME = os.getenv("MODEL_NAME", "pretrained_transformer")

CKPT_PATHS = {
    "pretrained_transformer": os.getenv(
        "CKPT_PRETRAINED",
        "/kaggle/input/datasets/flixwg/aml2026-model-weights/best_pretrained_transformer_kaggle.pt",
    ),
    "cnn_transformer": os.getenv(
        "CKPT_CNN_TRANSFORMER",
        "/kaggle/input/datasets/flixwg/aml2026-model-weights/best_cnn_transformer_kaggle.pt",
    ),
}

# Auto-detect competition data root — tries known Kaggle mount points in order.
# Ignores DATA_ROOT env var if it points at a directory without test_soundscapes/.
def _find_data_root() -> str:
    candidates = [
        os.getenv("DATA_ROOT", ""),
        "/kaggle/input/birdclef-2026",
        "/kaggle/input/competitions/birdclef-2026",
    ]
    for c in candidates:
        if c and glob.glob(os.path.join(c, "test_soundscapes", "*.ogg")):
            return c
    # No .ogg files found (notebook save / dummy run) — fall back to dir check
    for c in candidates:
        if c and os.path.isdir(os.path.join(c, "test_soundscapes")):
            return c
    return "/kaggle/input/birdclef-2026"

DATA_ROOT  = _find_data_root()
TEST_DIR   = os.getenv("TEST_DIR",   os.path.join(DATA_ROOT, "test_soundscapes"))
SAMPLE_SUB = os.getenv("SAMPLE_SUB", os.path.join(DATA_ROOT, "sample_submission.csv"))
OUTPUT     = os.getenv("OUTPUT",     "/kaggle/working/submission.csv")
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "16"))  # windows per forward pass
DEVICE     = torch.device("cpu")

assert MODEL_NAME in CKPT_PATHS, f"Unknown MODEL_NAME={MODEL_NAME!r}. Choose: {list(CKPT_PATHS)}"

print(f"MODEL_NAME : {MODEL_NAME}")
print(f"Checkpoint : {CKPT_PATHS[MODEL_NAME]}")
print(f"Data root  : {DATA_ROOT}")
print(f"Test dir   : {TEST_DIR}")
print(f"Output     : {OUTPUT}")
print(f"Batch size : {BATCH_SIZE}")

MODEL_NAME : pretrained_transformer
Checkpoint : /kaggle/input/datasets/flixwg/aml2026-model-weights/best_pretrained_transformer_kaggle.pt
Data root  : /kaggle/input/competitions/birdclef-2026
Test dir   : /kaggle/input/competitions/birdclef-2026/test_soundscapes
Output     : /kaggle/working/submission.csv
Batch size : 16


In [3]:
# ── Cell 3: Mel preprocessing constants ──────────────────────────────────────
#
# Defaults match data/preprocessing/data_pipeline.py exactly.
# Only override if using a checkpoint trained with different values.
#
SAMPLE_RATE   = int(os.getenv("SAMPLE_RATE",   "32000"))
CLIP_DURATION = int(os.getenv("CLIP_DURATION", "5"))      # seconds
N_MELS        = int(os.getenv("N_MELS",        "128"))
HOP_LENGTH    = int(os.getenv("HOP_LENGTH",    "512"))
N_FFT         = int(os.getenv("N_FFT",         "1024"))
F_MIN         = int(os.getenv("F_MIN",         "50"))
F_MAX         = int(os.getenv("F_MAX",         "14000"))

N_WINDOWS     = 12                              # BirdCLEF 2026: every soundscape is exactly 60s
FILE_SAMPLES  = SAMPLE_RATE * 60                # 60s of audio

print(f"SAMPLE_RATE={SAMPLE_RATE}, CLIP_DURATION={CLIP_DURATION}s")
print(f"N_MELS={N_MELS}, N_FFT={N_FFT}, HOP_LENGTH={HOP_LENGTH}")
print(f"F_MIN={F_MIN}, F_MAX={F_MAX}")
print(f"N_WINDOWS={N_WINDOWS}, FILE_SAMPLES={FILE_SAMPLES}")

SAMPLE_RATE=32000, CLIP_DURATION=5s
N_MELS=128, N_FFT=1024, HOP_LENGTH=512
F_MIN=50, F_MAX=14000
N_WINDOWS=12, FILE_SAMPLES=1920000


In [4]:
# ── Cell 4: Model definitions (inlined — no repo import) ─────────────────────

class PretrainedTransformer(nn.Module):
    """ViT-Small fine-tuned on mel spectrograms.
    Source: models/pretrained_transformer.py
    """
    def __init__(self, num_classes, *, vit_model="vit_small_patch16_224",
                 drop_path_rate=0.1, drop_rate=0.1, **_ignored):
        super().__init__()
        self.vit = timm.create_model(
            vit_model,
            in_chans=1,
            img_size=(128, 320),
            num_classes=num_classes,
            pretrained=False,   # no internet download at inference
            drop_path_rate=drop_path_rate,
            drop_rate=drop_rate,
        )

    def forward(self, x):
        x = F.pad(x, (0, 7))  # (B,1,128,313) → (B,1,128,320) to fit ViT patch grid
        return self.vit(x)


class CNNTransformer(nn.Module):
    """ResNet18 CNN front-end + Transformer encoder.
    Source: models/cnn_transformer.py
    """
    def __init__(self, num_classes, *, num_cnn_blocks=3, d_model=256,
                 n_heads=8, n_layers=4, dropout=0.1,
                 cnn_backbone="resnet18", pretrained_cnn=False, **_ignored):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError(f"d_model={d_model} must be divisible by n_heads={n_heads}")
        out_idx = num_cnn_blocks - 1

        self.cnn = timm.create_model(
            cnn_backbone, in_chans=1, features_only=True,
            out_indices=(out_idx,), pretrained=pretrained_cnn,
        )
        # probe spatial dimensions without backbone-specific tables
        with torch.no_grad():
            _feat = self.cnn(torch.zeros(1, 1, 128, 313))[0]
        _, cnn_channels, H, W = _feat.shape

        self.proj      = nn.Conv2d(cnn_channels, d_model, kernel_size=1, bias=False)
        self.proj_norm = nn.BatchNorm2d(d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, d_model, H, W))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model, num_classes),
        )

    def forward(self, x):
        B = x.shape[0]
        feat = self.cnn(x)[0]
        feat = self.proj_norm(self.proj(feat)) + self.pos_embed
        feat = feat.flatten(2).transpose(1, 2)          # (B, H*W, d_model)
        cls  = self.cls_token.expand(B, -1, -1)
        tok  = self.transformer(torch.cat([cls, feat], dim=1))
        return self.head(self.norm(tok)[:, 0])           # classify from [CLS]


def build_model(ckpt: dict) -> nn.Module:
    """Reconstruct model from self-contained checkpoint dict."""
    name        = ckpt["model_name"]
    num_classes = ckpt["num_classes"]
    kwargs      = {k: v for k, v in ckpt.get("model_kwargs", {}).items()
                   if k != "num_classes"}

    if name == "pretrained_transformer":
        return PretrainedTransformer(num_classes, **kwargs)
    elif name == "cnn_transformer":
        kwargs["pretrained_cnn"] = False   # architecture only — weights loaded from ckpt
        return CNNTransformer(num_classes, **kwargs)
    else:
        raise ValueError(f"Unknown model_name in checkpoint: {name!r}")


print("Model classes defined: PretrainedTransformer, CNNTransformer")

Model classes defined: PretrainedTransformer, CNNTransformer


In [5]:
# ── Cell 5: Load checkpoint and build model ───────────────────────────────────
ckpt_path = CKPT_PATHS[MODEL_NAME]
assert os.path.exists(ckpt_path), (
    f"Checkpoint not found: {ckpt_path}\n"
    "Upload the .pt file as a Kaggle dataset named 'aml2026-model-weights' "
    "and attach it to this notebook (see Cell 0 instructions)."
)

ckpt   = torch.load(ckpt_path, map_location="cpu", weights_only=False)
model  = build_model(ckpt)

state  = ckpt["model_state"]
# Strip _orig_mod. prefix saved when checkpoint came from a torch.compile() model
if any(k.startswith("_orig_mod.") for k in state):
    state = {k.removeprefix("_orig_mod."): v for k, v in state.items()}

model.load_state_dict(state)
model.eval()
model.to(DEVICE)

classes = ckpt["classes"]   # list[str]: K species sorted — 206 (report) or 234 (kaggle track)

ckpt_metric = ckpt.get("checkpoint_metric", "val_auc")
track       = ckpt.get("track", "report")

print(f"Loaded     : {ckpt['model_name']}  (track={track})")
print(f"Epoch      : {ckpt.get('epoch', '?')}")
print(f"Val AUC    : {ckpt.get('val_auc', float('nan')):.4f}")
if "sc_auc" in ckpt and not (isinstance(ckpt['sc_auc'], float) and ckpt['sc_auc'] != ckpt['sc_auc']):
    print(f"SC  AUC    : {ckpt['sc_auc']:.4f}  (checkpoint_metric={ckpt_metric})")
print(f"Num classes: {len(classes)}  (K={'234 taxonomy' if len(classes) == 234 else len(classes)})")
print(f"First 5    : {classes[:5]}")

Loaded     : pretrained_transformer  (track=kaggle)
Epoch      : 11
Val AUC    : 0.9452
SC  AUC    : 0.7932  (checkpoint_metric=sc_auc)
Num classes: 234  (K=234 taxonomy)
First 5    : ['1161364', '116570', '1176823', '1491113', '1595929']


In [6]:
# ── Cell 6: Spectrogram + inference helpers ───────────────────────────────────

def waveform_to_spec(y: np.ndarray) -> torch.Tensor:
    """5-second waveform → (1, N_MELS, T) tensor. Zero-pads short input."""
    target = SAMPLE_RATE * CLIP_DURATION
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)), mode="constant")
    else:
        y = y[:target]
    mel    = librosa.feature.melspectrogram(
        y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    return torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0)  # (1, N_MELS, T)


@torch.no_grad()
def predict_soundscape(model: nn.Module, file_path: str,
                        batch_size: int = BATCH_SIZE) -> dict:
    """
    Pad/truncate soundscape to 60s, chunk into 12 non-overlapping 5s windows.
    Run batched forward passes and return per-window sigmoid probabilities.

    Returns:
        dict mapping end_second (int) → probs np.ndarray shape (K,)
        e.g. {5: array([0.1, ...]), 10: array([0.2, ...]), ...}
    """
    y, _ = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:
        y = y[:FILE_SAMPLES]

    target = SAMPLE_RATE * CLIP_DURATION
    specs = [
        waveform_to_spec(y[i * target : (i + 1) * target])
        for i in range(N_WINDOWS)
    ]

    all_probs = []
    for start in range(0, N_WINDOWS, batch_size):
        batch = torch.stack(specs[start : start + batch_size]).to(DEVICE)
        probs = torch.sigmoid(model(batch)).cpu().numpy()
        all_probs.append(probs)
    all_probs = np.concatenate(all_probs, axis=0)   # (N_WINDOWS, K)

    return {(i + 1) * CLIP_DURATION: all_probs[i] for i in range(N_WINDOWS)}


print("Helpers defined: waveform_to_spec, predict_soundscape")

Helpers defined: waveform_to_spec, predict_soundscape


In [7]:
# ── Cell 7: Build submission.csv ──────────────────────────────────────────────
test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.ogg")))
print(f"Test soundscapes: {len(test_files)}")

if len(test_files) == 0:
    # Use recursive=True and '**/*.ogg' to match files inside subfolders
    fallback = sorted(glob.glob(os.path.join(DATA_ROOT, "train_soundscapes", "**", "*.ogg"), recursive=True)) 
    
    # Just grab the first 10 to keep the dummy save run fast
    fallback = fallback[:10] 
    
    if fallback: 
        print(f"WARNING: test_soundscapes empty — using {len(fallback)} train file(s) as dummy (notebook save run)")
        test_files = fallback
    else:
        raise FileNotFoundError(
            f"No .ogg files in {TEST_DIR} and no train_soundscapes fallback found.\n"
            "Please check your DATA_ROOT pathing."
        )

rows = []
for fp in tqdm(test_files, desc="Soundscapes"):
    stem  = Path(fp).stem
    preds = predict_soundscape(model, fp)
    for end_sec, probs in preds.items():
        row = {"row_id": f"{stem}_{end_sec}"}
        for cls, p in zip(classes, probs):
            row[cls] = float(p)
        rows.append(row)

sub = pd.DataFrame(rows)

# align column order to sample_submission.csv
if os.path.exists(SAMPLE_SUB):
    sample_cols = pd.read_csv(SAMPLE_SUB, nrows=0).columns.tolist()
    sub = sub.reindex(columns=sample_cols).fillna(0.0)
    print(f"Aligned to sample_submission columns ({len(sample_cols)} cols)")
else:
    print("WARNING: sample_submission.csv not found — using checkpoint class order")

expected_rows = len(test_files) * N_WINDOWS
assert len(sub) == expected_rows, f"Expected {expected_rows} rows, got {len(sub)}"

out_dir = os.path.dirname(OUTPUT)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
sub.to_csv(OUTPUT, index=False)
print(f"Saved {len(sub)} rows → {OUTPUT}")

Test soundscapes: 0


Soundscapes: 100%|██████████| 10/10 [00:31<00:00,  3.19s/it]

Aligned to sample_submission columns (235 cols)
Saved 120 rows → /kaggle/working/submission.csv


In [8]:
# ── Cell 8: Sanity check ──────────────────────────────────────────────────────
prob_cols = [c for c in sub.columns if c != "row_id"]

print(f"Shape      : {sub.shape}")
print(f"Rows       : {len(sub)}")
print(f"NaN count  : {sub.isnull().sum().sum()}")
print(f"Prob range : [{sub[prob_cols].min().min():.4f}, {sub[prob_cols].max().max():.4f}]")
print(f"row_id sample: {sub['row_id'].iloc[0]}  →  {sub['row_id'].iloc[-1]}")

sub.head()

Shape      : (120, 235)
Rows       : 120
NaN count  : 0
Prob range : [0.0113, 0.9794]
row_id sample: BC2026_Train_0001_S08_20250606_030007_5  →  BC2026_Train_0010_S09_20250828_000000_60


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0001_S08_20250606_030007_5,0.069921,0.090377,0.044741,0.031991,0.048325,0.048419,0.046572,0.061966,0.042407,...,0.025491,0.047391,0.052827,0.026254,0.047521,0.035400,0.044277,0.054085,0.049098,0.029275
1,BC2026_Train_0001_S08_20250606_030007_10,0.070877,0.094579,0.044257,0.029495,0.046389,0.047612,0.046223,0.063935,0.042191,...,0.021933,0.053766,0.047508,0.022938,0.048574,0.035343,0.044414,0.042355,0.047623,0.039574
2,BC2026_Train_0001_S08_20250606_030007_15,0.070841,0.094464,0.045953,0.032285,0.047926,0.048940,0.047215,0.066426,0.043275,...,0.023631,0.060548,0.052430,0.025147,0.049170,0.034359,0.044292,0.048927,0.049964,0.034540
3,BC2026_Train_0001_S08_20250606_030007_20,0.070026,0.098354,0.044507,0.030439,0.046529,0.047839,0.046236,0.062849,0.042557,...,0.020062,0.047542,0.046135,0.024882,0.048583,0.037627,0.039163,0.046653,0.048876,0.036850
4,BC2026_Train_0001_S08_20250606_030007_25,0.068398,0.095010,0.043502,0.029725,0.046141,0.046381,0.044938,0.060824,0.042562,...,0.023422,0.041072,0.049097,0.025890,0.046744,0.032130,0.044023,0.044956,0.047916,0.031019
